# Imports

In [1]:
import os
import sys
import glob
import json
import math
import time
import pdb
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
from pathlib import Path
from sklearn.utils import resample
sys.path.append('/mnt/scripts/biliseq_he_class2')
from helpers import anno as annoHelper
%load_ext autoreload
%autoreload 2

model_path = Path('/mnt/data/biliseq_he_class/models')
results = Path('/mnt/results')
sampleinfo = Path('/mnt/sampleinfo')
v6_path=Path('/mnt/data/biliseq_he_class/proc/v6/') #flatten to /mnt/data/proc/v6
tiles_path=v6_path.joinpath('tiles') # this should be updated to tiles_500px!
raw = Path('/mnt/data/biliseq_he_class/raw/new_data_20220829') #When all the wrinkles are worked out, flatten to /mnt/data/raw
svs = [str(x.parts[-1]) for x in raw.glob('*.svs')]
anno = Path('/mnt/data/biliseq_he_class/annotations/') #result? sampleinfo? flatten
# info = pd.read_csv(sampleinfo.joinpath('new_data_20220829_diagnosis_data_with_anno_v3.tsv'),
#                                       sep = '\t')
info = pd.read_csv(sampleinfo.joinpath('new_data_20220829_diagnosis_data_438_with_anno_v4.tsv'),
                                      sep = '\t')

models = Path('/mnt/results/v8/models')
# use_model = 'densenet169_1fold_1rep_10933bal_224px'
use_model = 'densenet169_1fold_1rep_20542bal_224px_v4'
thresh = 10
fn = results.joinpath('malig_v8_224px_tiles_gt_%d_overlap_v2.tsv' % thresh) 
malig = pd.read_csv(fn, sep = '\t')

# slide_df = pd.read_csv(sampleinfo.joinpath('slide_df_v8.0_44.tsv'),sep='\t').drop(columns=['Unnamed: 0'])
# tile_df = pd.read_csv(sampleinfo.joinpath('tile_df_v8.0_256400.tsv'),sep='\t')# slide_df_v8.0_44_v2.tsv
slide_df = pd.read_csv(sampleinfo.joinpath('slide_df_v8.0_76_v4.tsv'),sep='\t').drop(columns=['Unnamed: 0'])
tile_df = pd.read_csv(sampleinfo.joinpath('tile_df_v8.0_298007_v4.tsv'),
                      dtype='str',
                      sep='\t')

fu_df = info.copy() # pd.read_csv(sampleinfo.joinpath('digital_log_bdb_cleaned_bi_435.tsv'),sep='\t')

# Generate expanded slide_df (remaining data)

In [3]:
idx = info.anno_class.isna() #Not annotated slides
test_set_slides = info.slide_num[idx].values
tt = fu_df.slide_num.isin(test_set_slides).values
valid_slide_df = fu_df.loc[tt,:].reset_index(drop=True)

valid_slide_df.loc[:,'class'] = valid_slide_df.pathologic_diagnosis.values == 'Adenocarcinoma'
valid_slide_df.loc[valid_slide_df.loc[:,'class'],'anno_class'] = 'malignant'
valid_slide_df.loc[~valid_slide_df.loc[:,'class'],'anno_class'] = 'benign'
valid_slide_df.loc[:,'slide'] = valid_slide_df.slide_num

temp = pd.Series(svs)
for i in range(0,valid_slide_df.shape[0]):
    slide = valid_slide_df.loc[i,'slide']
    idx = temp.str.contains(str(slide)).values
    valid_slide_df.loc[i,'raw_fn'] = temp[idx].values[0]

valid_slide_df.loc[:,'group'] = valid_slide_df.anno_class
fn = sampleinfo.joinpath('test_slide_df_v8.0_%d_v4.tsv' % valid_slide_df.shape[0])
print(fn)
valid_slide_df.to_csv(fn,sep = '\t')
print(valid_slide_df.shape)
# valid_slide_df.loc[:,['accession_number','raw_fn','slide']].head()
valid_slide_df.head()

/mnt/sampleinfo/test_slide_df_v8.0_362_v4.tsv
(362, 19)


,accession_number,pathologic_diagnosis,dob,location,follow_up_diagnosis,notes,phs,section_num,slide_num,fu_pos,path_pos,anno_fn,anno_class,raw_fn,anno_pnfn,tile_pn,class,slide,group
0,EAS20-6314 - 1013820,Adenocarcinoma,7/28/1958,Distal,Duodenal Adenocarcinoma,NaN,EAS20-6314,NaN,1013820,True,True,NaN,malignant,EAS20-6314 - 1013820.svs,NaN,NaN,True,1013820,malignant
1,MYS20-128 - 1 - 1013755,Adenocarcinoma,6/10/1937,Mid,Intrahepatic Cholangiocarcinoma,NaN,MYS20-128,1.0,1013755,True,True,NaN,malignant,MYS20-128 - 1 - 1013755.svs,NaN,NaN,True,1013755,malignant
2,MYS20-128 - 2 -1013757,Adenocarcinoma,6/10/1937,Mid,Intrahepatic Cholangiocarcinoma,NaN,MYS20-128,2.0,1013757,True,True,NaN,malignant,MYS20-128 - 2 - 1013757.svs,NaN,NaN,True,1013757,malignant
3,MYS20-128 - 3 - 1013756,Adenocarcinoma,6/10/1937,Mid,Intrahepatic Cholangiocarcinoma,NaN,MYS20-128,3.0,1013756,True,True,NaN,malignant,MYS20-128 - 3 - 1013756.svs,NaN,NaN,True,1013756,malignant
4,MYS20-293 - 1013761,Negative for Adenocarcinoma,3/12/1959,Distal,Benign Stricture,NaN,MYS20-293,NaN,1013761,False,False,NaN,benign,MYS20-293 - 1013761.svs,NaN,NaN,False,1013761,benign


# Also generate version that is 100 slides (cases?)

In [18]:
print(np.sum(valid_slide_df.group.str.contains('malig')),
      np.sum(valid_slide_df.group.str.contains('benign')),
      np.sum(valid_slide_df.group.isna()))

111 251 0


In [26]:
strat_by = valid_slide_df.phs.values

print(len(valid_slide_df.phs.unique()),
     np.sum(valid_slide_df.group.str.contains('malig')),
     valid_slide_df.shape[0])
df2 =  resample(valid_slide_df,
                random_state=42,
                replace=False,
                n_samples=100,
                stratify = strat_by)
print(len(df2.phs.unique()),
      df2.shape[0],
     df2.path_pos.sum())
ncase = len(df2.phs.unique())
fn = sampleinfo.joinpath('rand_valid_%dslides_%dcases_v4.tsv' % (df2.shape[0],
                                                          ncase))
print(fn)
df2.to_csv(fn,sep = '\t')

305 111 362
97 100 38
/mnt/sampleinfo/rand_valid_100slides_97cases_v4.tsv


# Look at SyntaxErrors

In [53]:
logs = models.joinpath(use_model).joinpath('logs').joinpath('infer')
fn = logs.joinpath('round_1_test_syntax_errors.txt')
edf = pd.read_csv(fn,sep=':',header=None)
slides = edf.iloc[:,0].str.split('_').str[1].str.split('.').str[0].to_list()
print(','.join(slides))

382,353,277,328,270,385,354,326,321,279,368,314,313,294,361,366,293,377,282,285,370,302,379,305,268,330,337,339,345,342,266,367,292,295,360,312,369,315,320,278,327,329,271,384,355,383,352,276,343,267,338,344,336,269,331,378,304,303,284,371,376,283,263,347,340,264,332,335,349,289,300,307,375,280,309,287,372,296,363,364,318,291,316,311,298,324,358,323,380,351,275,272,387,356,286,373,374,281,308,306,288,301,334,348,333,341,265,262,346,273,386,357,381,350,274,322,325,359,310,299,317,365,319,290,297,362


In [57]:
for n in slides:
    bad_fn = logs.joinpath('testset').joinpath('slurm-1107527_%s.out' % n)
    if bad_fn.exists():
        base = bad_fn.parts[-1]
        up = bad_fn.parent.joinpath('err')
        # print(str(up.joinpath(base)))
        os.rename(str(bad_fn),str(up.joinpath(base)))


# Look at ValeErrors

In [7]:
logs = models.joinpath(use_model).joinpath('logs').joinpath('infer')
fn = logs.joinpath('ValueError_log.txt')
valid_slide_df = pd.read_csv('/mnt/sampleinfo/test_slide_df_v8.0_388.tsv',
                           sep = '\t',
                          )
edf = pd.read_csv(fn,sep=':',
                  usecols=[0],
                  header=None)
slides = edf.iloc[:,0].str.split('_').str[1].str.split('.').str[0].to_list()
print(','.join(slides))

41,41,193,193,38,38,206,206


In [9]:
valid_slide_df.loc[np.unique([int(x) for x in slides]),:]

,Unnamed: 0,accession_number,pathologic_diagnosis,dob,location,follow_up_diagnosis,notes,phs,section_num,slide_num,fu_pos,path_pos,class,anno_class,slide,raw_fn,group
38,38,PHS15-42791 - 1011638,Negative for Adenocarcinoma,1947-07-25,Distal,Benign Stricture,NaN,PHS15-42791,NaN,1011638,False,False,False,benign,1011638,PHS15-44776 - 1011638.svs,benign
41,41,PHS15-44776 - 1011638,Negative for Adenocarcinoma,1944-06-03,Distal,Benign Stricture,PSC,PHS15-44776,NaN,1011638,False,False,False,benign,1011638,PHS15-44776 - 1011638.svs,benign
193,193,PHS19-27841 - 1013788,Negative for Adenocarcinoma,1975-06-06,Hilar,Benign Stricture,PSC,PHS19-27841,NaN,1013788,False,False,False,benign,1013788,PHS19-33986 - 1013788.svs,benign
206,206,PHS19-33986 - 1013788,Negative for Adenocarcinoma,1933-10-10,Distal,Pancreatic Ductal Adenocarcinoma,NaN,PHS19-33986,NaN,1013788,True,False,False,benign,1013788,PHS19-33986 - 1013788.svs,benign
